## 1. Import Libraries & Load Dataset

**Tujuan:** Memuat dataset hasil feature selection (`stroke_selected_features.csv`) dari NB05 sebagai basis persiapan data sebelum modeling (encoding, train-test split, penanganan imbalance).

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/processed/stroke_selected_features.csv')
print(f"Shape: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Kolom: {df.columns.tolist()}")

Shape: 5094 baris, 7 kolom
Kolom: ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease', 'smoking_status', 'stroke']


**Insight:** Dataset hasil feature selection berhasil dimuat (5.094 baris, 7 kolom) — 6 fitur final beserta target `stroke`, siap masuk tahap persiapan modeling.

## 2. Encoding Kategorikal

**Tujuan:** Mengubah `smoking_status` (satu-satunya kolom kategorikal tersisa) menjadi format numerik lewat one-hot encoding, supaya siap dipakai baik untuk Random Forest (NB09) maupun Logistic Regression (NB10).

In [ ]:
print(f"Kolom sebelum encoding: {df.columns.tolist()}")
print(f"Kategori smoking_status: {df['smoking_status'].unique().tolist()}")

df_encoded = pd.get_dummies(df, columns=['smoking_status'], drop_first=True)

dummy_cols = [col for col in df_encoded.columns if col.startswith('smoking_status_')]
df_encoded[dummy_cols] = df_encoded[dummy_cols].astype(int)

print(f"\nKolom sesudah encoding: {df_encoded.columns.tolist()}")
df_encoded.head()

Kolom sebelum encoding: ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease', 'smoking_status', 'stroke']
Kategori smoking_status: ['formerly smoked', 'never smoked', 'smokes', 'Unknown']

Kolom sesudah encoding: ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease', 'stroke', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes']


,age,avg_glucose_level,bmi,hypertension,heart_disease,stroke,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,67.0,228.69,36.6,0,1,1,1,0,0
1,80.0,105.92,32.5,0,1,1,0,1,0
2,49.0,171.23,34.4,0,0,1,0,0,1
3,79.0,174.12,24.0,1,0,1,0,1,0
4,81.0,186.21,29.0,0,0,1,1,0,0


**Insight:** `smoking_status` berhasil di-encode menjadi 3 kolom dummy (1 kategori dijadikan baseline implisit lewat `drop_first=True`). Total kolom bertambah dari 7 menjadi 9 (6 fitur asli - 1 kolom smoking_status + 3 kolom dummy + target).

## 3. Train-Test Split

**Tujuan:** Memisahkan data menjadi set training (untuk melatih model) dan testing (untuk evaluasi akhir yang jujur), **sebelum** penanganan imbalance apapun diterapkan.

**Kenapa split dilakukan sebelum handling imbalance:** Test set harus mencerminkan kondisi dunia nyata (termasuk imbalance-nya) supaya evaluasi model representatif. Kalau `class_weight`/SMOTE diterapkan ke seluruh data SEBELUM split, ada risiko *data leakage* — informasi dari data yang seharusnya "belum pernah dilihat" model (test set) ikut memengaruhi proses training. Prinsip ini terutama krusial kalau nanti kamu bereksperimen dengan SMOTE (karena SMOTE membuat data sintetis berdasarkan seluruh data) — untuk `class_weight` risikonya lebih kecil (karena tidak mengubah data), tapi urutan ini tetap praktik standar yang benar untuk diikuti.

In [3]:
X = df_encoded.drop(columns=['stroke'])
y = df_encoded['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} baris")
print(f"Test set : {X_test.shape[0]} baris")

print(f"\nProporsi stroke di train: {y_train.mean()*100:.2f}%")
print(f"Proporsi stroke di test : {y_test.mean()*100:.2f}%")

Train set: 4075 baris
Test set : 1019 baris

Proporsi stroke di train: 4.83%
Proporsi stroke di test : 4.81%


## 4. Handle Imbalance

**Tujuan:** Mengimplementasikan penanganan imbalance secara nyata menggunakan `class_weight='balanced'` — beda dari NB04 yang baru sebatas menganalisis dampaknya (accuracy paradox).

**Catatan penting soal cara kerja `class_weight`:** Berbeda dari SMOTE yang membuat data sintetis (mengubah `X_train`/`y_train` secara fisik), `class_weight` **tidak mengubah data sama sekali**. Parameter ini nanti dipasang langsung saat inisialisasi model di NB09 (Random Forest) dan NB10 (Logistic Regression) — bukan di notebook ini. Section ini menghitung angka bobotnya secara eksplisit, supaya kita tahu persis apa yang terjadi "di balik layar" ketika nanti cuma menulis `class_weight='balanced'` di kode modeling.

In [4]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

print("Distribusi kelas di training set:")
print(y_train.value_counts())

classes = np.array([0, 1])
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))

print(f"\nBobot yang dihitung untuk masing-masing kelas:")
for cls, w in class_weight_dict.items():
    label = "Tidak Stroke" if cls == 0 else "Stroke"
    print(f"  Kelas {cls} ({label}): {w:.4f}")

Distribusi kelas di training set:
stroke
0    3878
1     197
Name: count, dtype: int64

Bobot yang dihitung untuk masing-masing kelas:
  Kelas 0 (Tidak Stroke): 0.5254
  Kelas 1 (Stroke): 10.3426


**Insight:** Bobot yang dihitung menunjukkan kelas stroke mendapat penalti sekitar 20x lebih berat dibanding kelas tidak-stroke saat training — sebanding dengan rasio imbalance yang ditemukan di NB04. Berbeda dari SMOTE, `X_train` dan `y_train` tetap 100% berisi data pasien asli, tidak ada satupun baris sintetis yang ditambahkan.

## 5. Verifikasi Hasil Preparation

**Tujuan:** Memastikan data training & testing sudah sepenuhnya siap dipakai untuk modeling — seluruh kolom numerik, tidak ada missing value, dan proporsi kelas tetap terjaga.

In [5]:
print("=== VERIFIKASI ===\n")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape : {y_test.shape}\n")

print("Tipe data seluruh kolom X_train:")
print(X_train.dtypes)

print(f"\nMissing value di X_train: {X_train.isnull().sum().sum()}")
print(f"Missing value di X_test : {X_test.isnull().sum().sum()}")

print(f"\nProporsi stroke - train: {y_train.mean()*100:.2f}% | test: {y_test.mean()*100:.2f}%")

=== VERIFIKASI ===

X_train shape: (4075, 8)
X_test shape : (1019, 8)
y_train shape: (4075,)
y_test shape : (1019,)

Tipe data seluruh kolom X_train:
age                               float64
avg_glucose_level                 float64
bmi                               float64
hypertension                        int64
heart_disease                       int64
smoking_status_formerly smoked      int64
smoking_status_never smoked         int64
smoking_status_smokes               int64
dtype: object

Missing value di X_train: 0
Missing value di X_test : 0

Proporsi stroke - train: 4.83% | test: 4.81%


**Insight:** Seluruh kolom pada X_train dan X_test terkonfirmasi numerik (int/float), tidak ada missing value tersisa, dan proporsi kelas stroke konsisten antara train dan test set. Data siap sepenuhnya untuk tahap modeling.

## 6. Simpan Dataset Final

**Tujuan:** Menyimpan X_train, X_test, y_train, y_test sebagai file terpisah, siap dipakai langsung oleh NB09 (Random Forest) dan NB10 (Logistic Regression) tanpa perlu mengulang proses encoding/split.

In [6]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("4 file berhasil disimpan di data/processed/:")
print("- X_train.csv, X_test.csv, y_train.csv, y_test.csv")

# Verifikasi baca ulang
check = pd.read_csv('../data/processed/X_train.csv')
print(f"\nVerifikasi X_train: {check.shape[0]} baris, {check.shape[1]} kolom")

4 file berhasil disimpan di data/processed/:
- X_train.csv, X_test.csv, y_train.csv, y_test.csv

Verifikasi X_train: 4075 baris, 8 kolom


**Insight:** Data modeling final berhasil disimpan (4.075 baris training, 1.019 baris testing), dengan encoding lengkap dan proporsi kelas terjaga lewat stratified split. Data siap dipakai langsung pada NB09 (Random Forest) dan NB10 (Logistic Regression), termasuk bobot kelas (`class_weight='balanced'`) yang akan diterapkan saat inisialisasi kedua model tersebut.